# Module 6 · Demo — RAG Pipeline

**From 0 to Agentic AI — DataHack Summit 2026**

In Module 2 our bare model **hallucinated** the PTO policy — it had never seen our docs. **RAG**
(Retrieval-Augmented Generation) fixes that: at query time we **retrieve** the relevant text and
**inject** it into the prompt, so the model answers from real, cited context — no retraining.

### The flow
**index** (chunk + embed docs) → **retrieve** (find similar chunks) → **inject** (add to prompt) → **answer**

We build each step on a tiny company-docs corpus, then chain them.

---
## Setup

In [1]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml.
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2" "langgraph>=1.0,<2" \
               "langchain-text-splitters>=0.3" "langchain-chroma>=0.2" "chromadb>=0.5"

ERROR: Could not find a version that satisfies the requirement langchain<2,>=1.2 (from versions: 0.0.1, 0.0.2, 0.0.3, 0.0.4, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10, 0.0.11, 0.0.12, 0.0.13, 0.0.14, 0.0.15, 0.0.16, 0.0.17, 0.0.18, 0.0.19, 0.0.20, 0.0.21, 0.0.22, 0.0.23, 0.0.24, 0.0.25, 0.0.26, 0.0.27, 0.0.28, 0.0.29, 0.0.30, 0.0.31, 0.0.32, 0.0.33, 0.0.34, 0.0.35, 0.0.36, 0.0.37, 0.0.38, 0.0.39, 0.0.40, 0.0.41, 0.0.42, 0.0.43, 0.0.44, 0.0.45, 0.0.46, 0.0.47, 0.0.48, 0.0.49, 0.0.50, 0.0.51, 0.0.52, 0.0.53, 0.0.54, 0.0.55, 0.0.56, 0.0.57, 0.0.58, 0.0.59, 0.0.60, 0.0.61, 0.0.63, 0.0.64, 0.0.65, 0.0.66, 0.0.67, 0.0.68, 0.0.69, 0.0.70, 0.0.71, 0.0.72, 0.0.73, 0.0.74, 0.0.75, 0.0.76, 0.0.77, 0.0.78, 0.0.79, 0.0.80, 0.0.81, 0.0.82, 0.0.83, 0.0.84, 0.0.85, 0.0.86, 0.0.87, 0.0.88, 0.0.89, 0.0.90, 0.0.91, 0.0.92, 0.0.93, 0.0.94, 0.0.95, 0.0.96, 0.0.97, 0.0.98, 0.0.99rc0, 0.0.99, 0.0.100, 0.0.101rc0, 0.0.101, 0.0.102rc0, 0.0.102, 0.0.103, 0.0.104, 0.0.105, 0.0.106, 0.0.107, 0.0.108, 0.0.109, 0.0

In [2]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

for key in ["OPENAI_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## Step 1 · The documents

Normally you'd load PDFs / Slack / wiki pages with a **document loader**. To keep the demo
self-contained we hand-write a few `Document`s — the rest of the pipeline is identical.

In [3]:
from langchain_core.documents import Document

docs = [
    Document(page_content="The billing service is owned by the Payments team. On-call lead: Sam. Escalate outages in #billing-oncall.", metadata={"source": "runbook", "team": "payments"}),
    Document(page_content="Employees get 28 days of paid time off (PTO) per year, plus public holidays. Requests go through the HR portal.", metadata={"source": "hr-policy", "team": "people"}),
    Document(page_content="Production deploys run weekdays at 9pm IST via the release bot. Rollbacks are one command: `deploy rollback <service>`.", metadata={"source": "runbook", "team": "platform"}),
    Document(page_content="Expense reports over $500 need manager approval before submission. Reimbursement takes 5-7 business days.", metadata={"source": "finance-policy", "team": "finance"}),
]
print(len(docs), "documents")

4 documents


---
## Step 2 · Chunk

Long docs are split into overlapping **chunks**. Chunk size &amp; overlap are the dials that make
or break retrieval: too big = noise, too small = lost meaning.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)
print(len(chunks), "chunks")
print(chunks[0].page_content)

4 chunks
The billing service is owned by the Payments team. On-call lead: Sam. Escalate outages in #billing-oncall.


---
## Step 3 · Embed &amp; store (ChromaDB)

Each chunk is turned into an **embedding** (a vector) and stored in **Chroma**. Similar meaning
→ nearby vectors, which is what makes semantic search work.

In [5]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(chunks, embeddings)
print("indexed", vectorstore._collection.count(), "chunks")

indexed 4 chunks


---
## Step 4 · Retrieve

Given a question, embed it and pull the **most similar** chunks. Notice we never told it the
word "PTO" — semantic search finds it anyway.

In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
hits = retriever.invoke("how much holiday do I get?")
for h in hits:
    print("-", h.page_content[:80], "  <=", h.metadata["source"])

- Employees get 28 days of paid time off (PTO) per year, plus public holidays. Req   <= hr-policy
- Expense reports over $500 need manager approval before submission. Reimbursement   <= finance-policy


---
## Step 5 · Inject &amp; answer

Put the retrieved chunks into the prompt as **context**, and ask the model to answer *only* from
it (and to cite sources). This is the whole point — the answer is **grounded**.

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

def answer(question: str) -> str:
    hits = retriever.invoke(question)
    context = "\n\n".join(f"[{h.metadata['source']}] {h.page_content}" for h in hits)
    msgs = [
        SystemMessage("Answer using ONLY the context. Cite the [source]. If unknown, say so."),
        HumanMessage(f"Context:\n{context}\n\nQuestion: {question}"),
    ]
    return llm.invoke(msgs).content

print(answer("How many PTO days do I get, and how do I request them?"))

You get 28 days of paid time off (PTO) per year, plus public holidays. To request PTO, you need to submit your request through the HR portal [hr-policy].


In [8]:
print(answer("Who is on-call for billing?"))

The on-call lead for billing is Sam. [runbook]


In [9]:
print(answer("What is the CEO's salary?"))   # not in the docs -> should say it doesn't know

The CEO's salary is not mentioned in the provided context.


---
## Key takeaways
- **RAG changes what the model *knows*** — retrieve relevant text, inject it, answer from it.
- **Chunking + embeddings** decide retrieval quality — most RAG failures are *retrieval* failures.
- Grounding the answer in retrieved context (and citing it) is what kills hallucination.

➡️ **Next (the project):** wrap this retriever as a **`@tool`** so the assistant can *decide* when
to search internal docs — that's **Knowledge Assistant v3**.